In [1]:
# import from 'glacier_space' repository
import sys

path_repo_glacier_space = '/Users/geoalxx/Python/glacier_space/'
sys.path.append(path_repo_glacier_space)

from scripts.ModelDataEvaluator import ModelDataEvaluator as MDE

In [2]:
import pandas as pd
import xarray as xr
import numpy as np
from datetime import datetime

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [3]:
cfg_file = path_repo_glacier_space + 'config/MDE_config_H3_UAV_iMet_generic.ini'

def get_MDE_for_config(loc, from_date, to_date, movement='up', skipmeters=0):
    cell_id = df_loc[df_loc['New Name'] == loc]['cell_id'].values[0]
    alt = df_loc[df_loc['New Name'] == loc]['Alt'].values[0]

    # update URLs
    icon_file = f'{icon_exp}_LES_51m_ml_{cell_id}.nc'
    uav_path = f'/Users/geoalxx/HEFEX III/Airdata/l4/{loc}/'

    MDE.update_config(cfg_file, 'DIRECTORIES', 'uav_imet', uav_path)
    MDE.update_config(cfg_file, 'FILES', 'icon_data_point', icon_file)
    MDE.update_config(cfg_file, 'ICON', 'from_date', from_date)
    MDE.update_config(cfg_file, 'ICON', 'to_date', to_date)
    MDE.update_config(cfg_file, 'UAV_IMET', 'location', loc)
    MDE.update_config(cfg_file, 'UAV_IMET', 'altitude', str(alt))
    MDE.update_config(cfg_file, 'UAV_IMET', 'movement', movement)
    MDE.update_config(cfg_file, 'UAV_IMET', 'skipmeters', str(skipmeters))

    return MDE(cfg_file, uav_imet=True)

def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

## Script settings

In [4]:
# selection of variables
sel_var = ['temp','theta', 'qv', 'wspd', 'wdir']
# sel_var = ['temp']

# location settings
sel_loc = ['Q255', 'Q257', 'Q267', 'Q267L', 'Q272', 'Q272L', 'Q272R']
#sel_loc = ['Q272', 'Q272L']

# time settings
from_date, to_date = '2025-08-07', '2025-08-10'

# UAV data settings
max_alt = 500
skipmeters = 3

# ICON settings
icon_nlev = 50     # lowest n ICON model levels
icon_exp = 'v370_2030'

## Load and prepare location and time dimensions

In [5]:
# load locations
df_loc = pd.read_csv(data_dir + 'H3_Coords_w_cells.csv')
df_loc = df_loc[df_loc['New Name'].isin(sel_loc)]
df_loc

,cell_id,topo,Original Name,New Name,Type,UAV Takeoff,Lat,Lon,Alt,Comment,Alt_DEM_2022
2,34946,2673.014160,Q2,Q257,UAV,x,46.809810,10.781460,2575.0,from iMet,2583.60
4,35316,2629.500732,Q1,Q255,UAV,NaN,46.811560,10.786100,2550.0,from iMet,2546.79
5,35693,2798.266357,Q5,Q272L,UAV,NaN,46.802640,10.768980,2702.0,from iMet,2718.42
6,35704,2798.971191,Q4,Q272,UAV,x,46.801750,10.771130,2720.0,from iMet,2726.18
16,37255,2801.379150,Q6,Q272R,UAV,x,46.801250,10.773500,2721.5,from iMet,2727.64
17,37400,2766.195557,Q3L,Q267L,UAV,NaN,46.806704,10.772575,2658.0,from iMet,2673.22
18,37435,2766.071533,Q3,Q267,UAV,x,46.805410,10.774720,2675.0,from iMet,2677.03


In [6]:
# prepare location dimensions
dim_loc = df_loc['New Name']
dim_loc_lat = np.array(df_loc['Lat'])
dim_loc_lon = df_loc['Lon']
dim_loc_cid = df_loc['cell_id'].astype(int)
dim_loc_ele_icon = df_loc['topo']
dim_loc_ele_uav = df_loc['Alt']

In [7]:
# prepare time dimension (full 'to_date' must be included hence +1 day)
dim_time = pd.date_range(pd.to_datetime(from_date,utc=True),pd.to_datetime(to_date,utc=True) + pd.Timedelta(days=1), freq="1h")

## Load ICON and UAV data using `ModelDataEvaluator` framework

In [8]:
# prepare for ICON data
icon_data_info = {}

icon_var_data = {}
for var in sel_var:
    icon_var_data[var] = []

icon_z_mc_data = []  # actual height values in m
dim_height_ml = None    # model level indices

# prepare for UAV data
uav_data_info = {}

uav_var_data = {}
uav_var_data_ml = {}
for var in sel_var:
    uav_var_data[var] = []
    uav_var_data_ml[var] = []

uav_alt_data = []
dim_height = np.arange(0, max_alt, 1)

# process all locations and variables
for loc in dim_loc:
    # initialize MDE for location (individual ICON extract file)
    mde = get_MDE_for_config(loc, from_date, to_date, skipmeters=skipmeters)

    # collect ICON height information
    z_mc = mde.icon.get_altitude_full_level()
    icon_z_mc_data.append(z_mc[-icon_nlev:])

    # collect UAV height information
    uav_ele_launch = mde.uav_imet.altitude
    uav_alt_tmp = np.arange(uav_ele_launch, uav_ele_launch + max_alt, 1)
    uav_alt_data.append(uav_alt_tmp)

    for var in sel_var:
        #--- ICON
        # get actual ICON data and variable info
        df_icon, info_icon = mde.icon.get_data_3D(var, from_date, to_date)

        # collect ICON data and info
        icon_var_data[var].append(df_icon[-icon_nlev:])
        icon_data_info[var] = info_icon

        # initialize ICON model level index as dimension
        if dim_height_ml is None:
            dim_height_ml = df_icon.index[-icon_nlev:]

        #--- UAV
        # get UAV data and variable info: original resolution and aggregated to ICON model level
        df_uav, info_uav = mde.uav_imet.get_data(var)
        df_uav_ml,_      = mde.get_aggregated_uav_data(mde.uav_imet, var, df_icon)

        # collect UAV data and info
        df_uav = df_uav.reindex(index=uav_alt_tmp, columns=dim_time) # extent original data to match time dimension and max height
        uav_var_data[var].append(df_uav)
        uav_var_data_ml[var].append(df_uav_ml[-icon_nlev:])
        uav_data_info[var] = info_uav

########################################################################
ModelDataEvaluator:
Loading configuration file: /Users/geoalxx/Python/glacier_space/config/MDE_config_H3_UAV_iMet_generic.ini

------------------------------------------------------------------------
ICONLoader:
Converting half levels to full levels for variable: w
ICON loaded for grid cell: 1; surface altitude: 2674m
Time range: 2025-08-05 00:00:00+00:00 to 2025-09-05 18:00:00+00:00
------------------------------------------------------------------------

------------------------------------------------------------------------
UAViMetLoader:
Loading UAV iMet data from: /Users/geoalxx/HEFEX III/Airdata/l4/Q257//HEFEX3_UAV_Q257_up_*.csv
Location: Q257 (up)
------------------------------------------------------------------------

########################################################################
########################################################################
ModelDataEvaluator:
Loading configuration f

## Build datasets

In [9]:
def create_data_array_2D(data, name, height_name, height_dim, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in dim_time]

    # re-arrange shape according to desired dimensions
    data_t = np.array(data).transpose(2, 1, 0)

    # create DataArray
    xr_data = xr.DataArray(
        data_t,
        coords={
            "time": ("time", time_naive, {"long_name": "Timestamp in UTC"}),
            height_name: (height_name, height_dim, {"long_name": "Height index"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "ele_icon": ("location", dim_loc_ele_icon, {"long_name": "Elevation on ICON grid", "units": "meters"}),
        },
        dims=["time", height_name, "location"],
        name=name,
        attrs=var_attrs,
    )
    return xr_data

def create_data_array_height_values(data, name, height_name, var_attrs):
    # re-arrange shape according to desired dimensions
    data_t = np.array(data).transpose(1, 0)

    # create DataArray
    xr_data = xr.DataArray(
        data_t,
        dims=[height_name, "location"],
        name=name,
        attrs=var_attrs,
    )
    return xr_data

### ICON

In [10]:
# create new dataset
ds_out = xr.Dataset()

# add 2D variable data to dataset
for var in sel_var:
    ds_out[var] = create_data_array_2D(icon_var_data[var], var, 'height_ml', dim_height_ml, icon_data_info[var])

# add model height information (full level center)
_, z_mc_info = mde.icon.data_grid.get_variable_info('z_mc')
ds_out['z_mc'] = create_data_array_height_values(icon_z_mc_data, 'z_mc', 'height_ml', z_mc_info)

# add ICON cell IDs
ds_out.coords['cid'] = ("location", dim_loc_cid, {"long_name": "ICON Cell ID of UAV location"})

# add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset at 51m resolution for HEFEX III UAV locations (IOP1)",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "experiment_id": f"hefex3/{icon_exp}/exp_R3B15_51m",
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Based on glacier outlines from OGGM SSP370, year 2030"
}

output_file = data_out_dir + f'H3_IOP1_ICON_{icon_exp}_UAV_locations_vertical_profile.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/H3_IOP1_ICON_v370_2030_UAV_locations_vertical_profile.nc.


In [11]:
ds_out

<xarray.Dataset> Size: 682kB
Dimensions:    (time: 97, height_ml: 50, location: 7)
Coordinates:
  * time       (time) datetime64[ns] 776B 2025-08-07 ... 2025-08-11
  * height_ml  (height_ml) int64 400B 101 102 103 104 105 ... 147 148 149 150
  * location   (location) object 56B 'Q257' 'Q255' 'Q272L' ... 'Q267L' 'Q267'
    lat        (location) float64 56B 46.81 46.81 46.8 46.8 46.8 46.81 46.81
    lon        (location) float64 56B 10.78 10.79 10.77 10.77 10.77 10.77 10.77
    ele_icon   (location) float64 56B 2.673e+03 2.63e+03 ... 2.766e+03 2.766e+03
    cid        (location) int64 56B 34946 35316 35693 35704 37255 37400 37435
Data variables:
    temp       (time, height_ml, location) float32 136kB 0.5463 0.86 ... 10.33
    theta      (time, height_ml, location) float32 136kB 311.2 311.3 ... 308.9
    qv         (time, height_ml, location) float32 136kB 1.208 1.137 ... 8.132
    wspd       (time, height_ml, location) float32 136kB 2.974 3.028 ... 2.181
    wdir       (time, height_ml, location) float32 136kB 247.3 250.4 ... 209.9
    z_mc       (height_ml, location) float32 1kB 3.924e+03 ... 2.771e+03
Attributes:
    title:          Sliced ICON-LES dataset at 51m resolution for HEFEX III U...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-04-14
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    campaign:       HEFEX III
    experiment_id:  hefex3/v370_2030/exp_R3B15_51m
    StartTime:      2025-08-07 00:00:00
    StopTime:       2025-08-11 00:00:00
    comment:        Based on glacier outlines from OGGM SSP370, year 2030

### Observations from UAV

In [12]:
# create new dataset
ds_out = xr.Dataset()

# add 2D variable data to dataset
for var in sel_var:
    ds_out[var] = create_data_array_2D(uav_var_data[var], var, 'height', dim_height, uav_data_info[var])
    ds_out[f'{var}_ml'] = create_data_array_2D(uav_var_data_ml[var], f'{var}_ml', 'height_ml', dim_height_ml, uav_data_info[var])

# add model height information (full level center)
ds_out['altitude'] = create_data_array_height_values(uav_alt_data, 'altitude', 'height', {})

# add global attributes
ds_out.attrs = {
    "title": f"Vertical atmospheric profiles from UAV soundings acquired during IOP1 of HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "Field measurements from DJI Quadcopters with attached iMET XQ-2 sensors",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Original UAV data was additionally aggregated to ICON model levels (*_ml). Data has not been corrected to account for altitude offsets."
}

output_file = data_out_dir + f'H3_IOP1_Observations_UAV_vertical_profile.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/H3_IOP1_Observations_UAV_vertical_profile.nc.


In [13]:
ds_out

<xarray.Dataset> Size: 15MB
Dimensions:    (time: 97, height: 500, location: 7, height_ml: 50)
Coordinates:
  * time       (time) datetime64[ns] 776B 2025-08-07 ... 2025-08-11
  * height     (height) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499
  * location   (location) object 56B 'Q257' 'Q255' 'Q272L' ... 'Q267L' 'Q267'
    lat        (location) float64 56B 46.81 46.81 46.8 46.8 46.8 46.81 46.81
    lon        (location) float64 56B 10.78 10.79 10.77 10.77 10.77 10.77 10.77
    ele_icon   (location) float64 56B 2.673e+03 2.63e+03 ... 2.766e+03 2.766e+03
  * height_ml  (height_ml) int64 400B 101 102 103 104 105 ... 147 148 149 150
Data variables:
    temp       (time, height, location) float64 3MB nan nan nan ... nan nan nan
    temp_ml    (time, height_ml, location) float64 272kB nan nan nan ... nan nan
    theta      (time, height, location) float64 3MB nan nan nan ... nan nan nan
    theta_ml   (time, height_ml, location) float64 272kB nan nan nan ... nan nan
    qv         (time, height, location) float64 3MB nan nan nan ... nan nan nan
    qv_ml      (time, height_ml, location) float64 272kB nan nan nan ... nan nan
    wspd       (time, height, location) float64 3MB nan nan nan ... nan nan nan
    wspd_ml    (time, height_ml, location) float64 272kB nan nan nan ... nan nan
    wdir       (time, height, location) float64 3MB nan nan nan ... nan nan nan
    wdir_ml    (time, height_ml, location) float64 272kB nan nan nan ... nan nan
    altitude   (height, location) float64 28kB 2.575e+03 2.55e+03 ... 3.174e+03
Attributes:
    title:        Vertical atmospheric profiles from UAV soundings acquired d...
    institution:  Humboldt-Universität zu Berlin
    source:       Field measurements from DJI Quadcopters with attached iMET ...
    history:      Created 2026-04-14
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX III
    StartTime:    2025-08-07 00:00:00
    StopTime:     2025-08-11 00:00:00
    comment:      Original UAV data was additionally aggregated to ICON model...